# Day 1: Talking to an LLM with LangChain

**LangChain Hands-on Series | Day 1 of 10**
Trainer: Ajeetkumar

---

Welcome! Before we touch any fancy agents or RAG systems, we need to get comfortable with one simple thing: **sending a message to an LLM and understanding what comes back**. Everything else in LangChain is built on top of this.

By the end of today you will be able to:

- set up your API key safely (no keys pasted inside notebooks, please)
- call an OpenAI model through LangChain and read its reply
- understand system, human and AI messages, and why the order matters
- control the model's creativity and answer length
- use `invoke`, `stream` and `batch` - the three ways of calling anything in LangChain

**How to work through this notebook:** run one cell at a time (Shift + Enter), read the note above it first, and whenever you see **Pause and predict**, actually stop and guess the answer before running the cell. It feels slow, but it is the fastest way to really learn this.

### The 10-day plan

Here is where we are going. Do not worry if some words look unfamiliar right now - each one gets its own day.

| Day | What we cover | What you will have built |
|-----|---------------|--------------------------|
| 1 | Chat models and messages | Your first LLM calls (today) |
| 2 | Prompt templates and structured output | A review analyser that returns clean data |
| 3 | LCEL: chaining steps together | A content creation pipeline |
| 4 | Memory | A chatbot that remembers you |
| 5 | Documents, embeddings, vector stores | Semantic search over company policies |
| 6 | RAG | A policy Q&A bot that cites its sources |
| 7 | Tools | An LLM that can call your Python functions |
| 8 | Agents | An assistant that decides on its own which tool to use |
| 9 | Production features | Async, caching, cost tracking, guardrails |
| 10 | Capstone | A complete HR and customer support assistant |

## Why do we even need LangChain?

Fair question. OpenAI already has a Python library. Why add another layer?

Think about it this way. Calling an LLM is like making a phone call - easy. But a real application is more like running a call centre: you need scripts (prompts), call notes (memory), a manual to look things up in (your documents), the ability to transfer the call to billing (tools), and a supervisor who decides who handles what (agents).

You *could* build all of that yourself. LangChain gives you tested building blocks for each piece, and they all plug into each other the same way.

```
                           YOUR APPLICATION
                                  |
     +----------+----------+------+-----+-----------+--------------+
     |          |          |            |           |              |
  Prompts    Models     Parsers      Memory    Retrievers    Tools & Agents
     |          |          |            |           |              |
     +----------+----------+------+-----+-----------+--------------+
                                  |
                   all connected with the  |  (pipe) operator
                                  |
         OpenAI / Anthropic / Google / Ollama / Azure ... any provider
```

One more big advantage: if your company moves from OpenAI to another provider tomorrow, you change **one line**, not your whole codebase.

### The packages you will see

| Package | What lives inside |
|---------|-------------------|
| `langchain-core` | The basics: messages, prompts, parsers, the Runnable interface |
| `langchain` | Higher-level features like `create_agent` and middleware |
| `langchain-openai` | Everything OpenAI: `ChatOpenAI`, `OpenAIEmbeddings` |
| `langchain-community` | Community-maintained integrations, mostly document loaders |
| `langgraph` | The engine that runs agents behind the scenes |

## Step 1: Install the packages

You only need to do this once. The `requirements.txt` file in this folder has everything for all 10 days, so you will not need to install anything again later.

In [ ]:
# The % in front of pip makes sure packages go into THIS notebook's Python, not some other one
# -q keeps the output short; -r tells pip to read the list from requirements.txt
%pip install -q -r requirements.txt

: 

## Step 2: Keep your API key in a `.env` file

A quick story: a lot of people have pushed notebooks to GitHub with their API key inside, and found a big bill a few days later because bots scan GitHub for keys. So we never put the key in the code.

Create a file called `.env` in this project folder and put this inside:

```
OPENAI_API_KEY=sk-your-key-here
OPENAI_MODEL=gpt-4.1-nano
```

This project's `.gitignore` already ignores `.env`, so it stays on your machine.

Here is what happens when we load it:

```
   .env file  --->  load_dotenv()  --->  environment variables  --->  ChatOpenAI picks up OPENAI_API_KEY by itself
```

In [ ]:
import os                        # lets us read environment variables
from dotenv import load_dotenv   # small library that reads .env files

load_dotenv()                    # finds the .env file and loads every KEY=VALUE line into the environment

In [ ]:
# If OPENAI_MODEL is missing from .env, we fall back to gpt-4.1-nano
MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-4.1-nano")

# We only check whether the key exists. Never print the key itself.
print("API key found? ", os.getenv("OPENAI_API_KEY") is not None)
print("Model we'll use:", MODEL_NAME)

If it says `False`, the most common reasons are: the file is named `.env.txt` by mistake (Windows likes to do this), or it is in a different folder from the notebook. Fix it, then restart the kernel.

> **About the model:** we use OpenAI's small "nano" model because it is fast and very cheap - perfect for learning. `gpt-4.1-nano` supports every feature used in this course. If you switch to a reasoning model like `gpt-5-nano`, delete the `temperature=` arguments, because those models do not accept a custom temperature.

## Step 3: Your first LLM call

`ChatOpenAI` is LangChain's wrapper around OpenAI's chat models. Creating it does **not** call the API - it just remembers your settings. The actual call happens when we use `.invoke()`.

```
  "In one sentence, what is LangChain?"
                 |
                 v
          llm.invoke(...)   ------>  OpenAI servers
                                          |
                 AIMessage  <-------------+
```

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=MODEL_NAME)   # just stores the settings - no API call yet
llm                                  # displaying it shows the configuration

In [ ]:
response = llm.invoke("In one sentence, what is LangChain?")   # this is the real API call
print(response.content)                                       # .content holds the text answer

That is it - you have talked to an LLM through LangChain. Now let's look more closely at what came back, because it is more than just text.

## Step 4: What is inside the response?

The reply is an `AIMessage` object. Think of it like an email: there is the message body, but there are also headers with extra details.

```
AIMessage
 |-- content             the answer text (the "body")
 |-- response_metadata   which model answered, why it stopped
 |-- usage_metadata      how many tokens were used  -> this is what you pay for
 |-- id                  unique id for this response
```

In [ ]:
print(type(response).__name__)   # the class name: AIMessage
print(response.content)          # the body

In [ ]:
meta = response.response_metadata          # a plain dictionary with provider details

print("Model that answered:", meta.get("model_name"))
print("Why it stopped     :", meta.get("finish_reason"))   # "stop" means it finished naturally

In [ ]:
usage = response.usage_metadata            # token counts for this single call

print("Tokens in my question :", usage["input_tokens"])
print("Tokens in the answer  :", usage["output_tokens"])
print("Total (what I pay for):", usage["total_tokens"])

**What is a token?** Roughly a piece of a word. In English, 100 tokens is about 75 words. Pricing is per million tokens, so a single small call costs a tiny fraction of a rupee - but it adds up when you have thousands of users.

Let's draw it. A picture makes it obvious which side costs more.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(5, 3))
plt.bar(["My question (input)", "Model answer (output)"],
        [usage["input_tokens"], usage["output_tokens"]],
        color=["#4C72B0", "#DD8452"])
plt.title("Where did the tokens go?")
plt.ylabel("Tokens")
plt.show()

## Step 5: Messages - who said what

Chat models do not really take "a string". They take a **conversation**: a list of messages, each with a role.

A good way to remember the roles is a stage play:

| Message | Role in the play | Written by |
|---------|------------------|------------|
| `SystemMessage` | The director's instructions to the actor ("you are a strict teacher") | You, the developer |
| `HumanMessage` | Lines spoken by the audience member | The user |
| `AIMessage` | The actor's reply | The model |
| `ToolMessage` | A note handed to the actor with information they asked for (Day 7) | Your code |

When we passed a plain string earlier, LangChain quietly wrapped it into a `HumanMessage` for us.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage(content="You are a friendly Python teacher. Answer in 2 short sentences for beginners."),
    HumanMessage(content="What is a Python list?"),
]

reply = llm.invoke(messages)   # we send the whole list, not just the question
print(reply.content)

**Pause and predict:** we will now keep the exact same question but change only the system message to "You are a pirate". What do you expect to change in the answer - the facts, the style, or both?

In [ ]:
pirate_messages = [
    SystemMessage(content="You are a pirate. Answer every question in pirate language in 2 sentences."),
    HumanMessage(content="What is a Python list?"),   # same question as before
]

print(llm.invoke(pirate_messages).content)

<details>
<summary>Click to see the explanation</summary>

The facts stay roughly the same (a list is still an ordered collection), but the style changes completely. The system message is the most powerful lever you have for controlling tone, format, length and rules. In real projects, a lot of the work is writing a good system message.
</details>

### Shorter ways to write messages

Typing `SystemMessage(content=...)` every time gets tiring. LangChain also accepts **tuples** and **dictionaries**. All three styles send exactly the same request - use whichever you find easiest to read.

In [ ]:
# Tuple style: (role, text)
print(llm.invoke([
    ("system", "You are a helpful assistant. Reply in one sentence."),
    ("human", "Name one benefit of using Python."),
]).content)

In [ ]:
# Dictionary style: same format as the raw OpenAI API, handy if you are migrating old code
print(llm.invoke([
    {"role": "system", "content": "You are a helpful assistant. Reply in one sentence."},
    {"role": "user", "content": "Name one benefit of using Python."},
]).content)

## Step 6: Having a conversation

Here is something that surprises almost every beginner: **the model has no memory at all**. Each call is completely fresh. ChatGPT feels like it remembers you only because the app sends the whole chat history every single time.

So to have a conversation, *we* keep a list and keep adding to it:

```
Call 1 sends:  [System, Human-1]                                ->  AI-1
Call 2 sends:  [System, Human-1, AI-1, Human-2]                 ->  AI-2
Call 3 sends:  [System, Human-1, AI-1, Human-2, AI-2, Human-3]  ->  AI-3
                 (the list keeps growing)
```

In [ ]:
conversation = [SystemMessage(content="You are a helpful assistant. Keep answers short.")]

conversation.append(HumanMessage(content="Hi, my name is Ajeetkumar and I teach AI."))
first_reply = llm.invoke(conversation)
conversation.append(first_reply)            # important: save the model's reply into the history too

print("AI:", first_reply.content)

In [ ]:
conversation.append(HumanMessage(content="What is my name and what do I teach?"))
second_reply = llm.invoke(conversation)     # the full history goes along with the question
conversation.append(second_reply)

print("AI:", second_reply.content)

In [ ]:
# Let's look at everything we actually sent on the last call
for m in conversation:
    print(f"{m.type:>6} | {m.content[:80]}")

**Pause and predict:** what happens if we ask "What is my name?" in a brand-new call, without the history?

In [ ]:
print(llm.invoke("What is my name and what do I teach?").content)

<details>
<summary>Click to see the explanation</summary>

The model has no idea. It will say something like "I don't have access to that information". This is the proof that LLMs are stateless. On Day 4 we will let LangChain manage this history for us, including what to do when it gets too long.
</details>

## Step 7: Two settings you will use all the time

### Temperature - how adventurous should the model be?

Imagine asking two friends for a restaurant suggestion. One always says "the usual place". The other suggests somewhere new every time. That is temperature.

- `temperature=0` - focused, predictable, nearly the same answer every time. Use it for extraction, classification, factual Q&A.
- `temperature=1` and above - creative and varied. Use it for brainstorming, names, stories.

In [ ]:
focused_llm = ChatOpenAI(model=MODEL_NAME, temperature=0)      # the "usual place" friend
creative_llm = ChatOpenAI(model=MODEL_NAME, temperature=1.2)   # the adventurous friend

prompt = "Suggest a catchy name for an AI training institute. Reply with the name only."

In [ ]:
for attempt in range(1, 4):
    print(f"Attempt {attempt}")
    print("   temperature 0   :", focused_llm.invoke(prompt).content)
    print("   temperature 1.2 :", creative_llm.invoke(prompt).content)

Notice how the temperature 0 answers barely change, while the 1.2 answers are different each time.

### max_tokens - put a limit on the answer length

Useful when you want short answers or want to protect your budget. The model simply stops when it reaches the limit, even mid-sentence.

In [ ]:
short_llm = ChatOpenAI(model=MODEL_NAME, max_tokens=20)    # roughly 15 words at most

short = short_llm.invoke("Explain the history of artificial intelligence in detail.")
print(short.content)
print("\nWhy it stopped:", short.response_metadata.get("finish_reason"))

`finish_reason = "length"` tells you the answer was cut off by `max_tokens`, not finished naturally. In a real app, it is worth checking this so you never show users a half-finished sentence without knowing it.

## Step 8: invoke, stream and batch

This is the most important idea of the day, so let's be clear about it. **Every** LangChain component - models, prompts, chains, retrievers, agents - can be called in the same three ways:

```
invoke(input)            one input   -> wait   -> one full output
stream(input)            one input   -> output arrives piece by piece (the ChatGPT "typing" effect)
batch([in1, in2, ...])   many inputs -> run in parallel -> list of outputs, same order
```

Learn these three once and you can use every component in the library.

### Streaming

Why bother? Because a user staring at a blank screen for 5 seconds feels like the app is broken. If the words start appearing after half a second, it feels fast - even when the total time is the same.

In [ ]:
for chunk in llm.stream("Give me 3 short tips for learning Python."):
    print(chunk.content, end="", flush=True)    # print each piece as it arrives, on the same line

What exactly are those "pieces"? Let's catch them in a list and look.

In [ ]:
chunks = [chunk for chunk in llm.stream("Say hello in 5 words.")]

print("Number of chunks:", len(chunks))
for c in chunks[:6]:
    print(repr(c.content))      # repr shows the exact text, including spaces

In [ ]:
# Chunks can be glued back together with +
full = chunks[0]
for c in chunks[1:]:
    full = full + c

print(full.content)

### Batch

If you have 50 product descriptions to summarise, do not write a `for` loop with `invoke()`. A loop waits for each call to finish before starting the next. `batch()` sends them in parallel.

In [ ]:
questions = [
    "What is Python? Answer in one sentence.",
    "What is machine learning? Answer in one sentence.",
    "What is LangChain? Answer in one sentence.",
]

answers = llm.batch(questions)          # all three go out at the same time

for q, a in zip(questions, answers):    # results come back in the same order as the questions
    print("Q:", q)
    print("A:", a.content, "\n")

**Pause and predict:** for 5 small requests, how much faster do you think `batch()` will be compared to a loop? 2 times? 5 times? Run the cell and see.

In [ ]:
import time

test_prompts = [f"Give me one word that rhymes with number {i}." for i in range(5)]

start = time.time()
for p in test_prompts:                  # one after another
    llm.invoke(p)
loop_seconds = time.time() - start

start = time.time()
llm.batch(test_prompts)                 # all together
batch_seconds = time.time() - start

print(f"Loop with invoke(): {loop_seconds:.2f} seconds")
print(f"One batch() call  : {batch_seconds:.2f} seconds")

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(["invoke() in a loop", "batch()"], [loop_seconds, batch_seconds], color=["#C44E52", "#55A868"])
plt.title("Same 5 requests, two approaches")
plt.ylabel("Seconds")
plt.show()

## Step 9: `init_chat_model` - switch providers in one line

So far we created `ChatOpenAI` directly. There is also a provider-neutral way: give it a string like `"openai:gpt-4.1-nano"`.

```
init_chat_model("openai:gpt-4.1-nano")       ->  ChatOpenAI
init_chat_model("anthropic:<model-name>")    ->  ChatAnthropic          (pip install langchain-anthropic)
init_chat_model("google_genai:<model-name>") ->  ChatGoogleGenerativeAI (pip install langchain-google-genai)
init_chat_model("ollama:llama3.2")           ->  ChatOllama, runs on your own laptop
```

If you keep the model string in your `.env` file, you can switch providers without touching the code at all.

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(f"openai:{MODEL_NAME}", temperature=0)
print(type(model).__name__)                                    # it is still a ChatOpenAI underneath
print(model.invoke("Reply with the single word: Ready").content)

## Step 10: Mini project - a Python tutor you can reuse

Let's put today's ideas together into a small function you could actually use: a fixed persona (system message), an adjustable level, and streaming output.

In [ ]:
def ask_tutor(question, level="beginner"):
    """Ask the Python tutor a question. The answer streams to the screen and is also returned."""
    system_text = (
        f"You are a patient Python tutor. The student is a {level}. "
        "Explain in at most 4 short sentences and include one tiny code example."
    )
    msgs = [SystemMessage(content=system_text), HumanMessage(content=question)]

    answer = ""
    for chunk in llm.stream(msgs):          # stream so the student sees the answer immediately
        print(chunk.content, end="", flush=True)
        answer += chunk.content             # also keep the full text so we can return it
    print()
    return answer

In [ ]:
_ = ask_tutor("What is a dictionary in Python?")

In [ ]:
_ = ask_tutor("What is a dictionary in Python?", level="senior developer")

### Your turn

Change the question and the level below and run it. Try a level like "10-year-old child" or "data scientist" and see how the explanation changes.

In [ ]:
# Edit these two lines and run the cell
my_question = "What is a for loop?"
my_level = "10-year-old child"

_ = ask_tutor(my_question, level=my_level)

## Common mistakes on Day 1

| What you see | What usually went wrong |
|--------------|-------------------------|
| `AuthenticationError` / 401 | Wrong key, or the `.env` file was not loaded. Restart the kernel after creating `.env`. |
| `API key found? False` | `.env` saved as `.env.txt`, or saved in a different folder |
| `NotFoundError` about the model | Typo in the model name, or your account has no access to it |
| `Unsupported value: temperature` | You are using a reasoning model (like `gpt-5-nano`). Remove `temperature`. |
| The model "forgot" what you said | You did not send the history. Models are stateless. |

## Quick recap

- `ChatOpenAI(...)` stores settings; `.invoke()` makes the call and returns an `AIMessage`.
- `.content` is the text; `.usage_metadata` tells you the token cost.
- Chat models take a list of messages. The system message controls behaviour.
- Models remember nothing - you send the history each time.
- `temperature` controls creativity; `max_tokens` controls length.
- `invoke`, `stream`, `batch` work on every LangChain component.

## Practice before Day 2

1. Write a system message that makes the model reply only in Hindi or Marathi, and ask it three questions.
2. Have a 4-turn conversation about your favourite movie, then print the full history.
3. Use `batch()` to translate 5 English sentences into French.
4. Write a function `show_cost(response)` that prints the input, output and total tokens of any response.
5. Ask for a 2-line poem with `temperature=0` and `temperature=1.2` three times each. Which one would you use for a product name generator?

**Tomorrow:** instead of writing prompts by hand every time, we will build reusable prompt templates and get clean, structured data back from the model.